In [7]:
import pandas as pd
import numpy as np
import geopandas as gpd
import transbigdata as tbd
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm  # 显示循环进度
import time

In [8]:
##### 1. 数据读取
# 数据路径
base_path = r'd:/Geo_python/Geo_data/2019/01/20190901'
# 充电站数据
charge_station = pd.read_csv('d:/Geo_python/Geo_data/2019/stations.csv', encoding='gbk')[['lon', 'lat']]
charge_station = charge_station.rename(columns={'lon': 'station_lon'})
charge_station = charge_station.rename(columns={'lat': 'station_lat'})
# 深圳范围数据
sz = gpd.read_file('sz.json')
# 车牌id
id_all = pd.read_csv(r'id_all.csv')
print(f'总车辆数: {len(id_all)}')

总车辆数: 21718


In [9]:
# 创建空列表用于存储所有充电事件
all_charge_events = []
# 创建一个字典用于统计每个车辆ID的充电次数
vehicle_charge_count = {}

In [10]:
start = time.perf_counter()  # 开始时间

# 遍历每个车辆ID
for index, row in tqdm(id_all.iterrows(), total=len(id_all), desc='处理车辆'):
    car_id = row['id']
    
    # 用于存储当前车辆的所有数据
    car_data_all = []
    
    # 循环读取每个出租车txt文件，提取当前车辆ID的数据
    for i in range(1, 289):
        # 读取文件
        filename = f"{i:03d}.txt"
        full_path = base_path + '_' + filename
        
        # 使用pandas的chunksize参数分块读取大文件，避免内存溢出
        for chunk in pd.read_csv(full_path, header=None, chunksize=100000):
            # 添加列名
            chunk.columns = ['date','date_time','H','id','lon','lat','speed','heading','OpenStatus','shift_type']
            # 筛选当前车辆ID的数据
            car_chunk = chunk[chunk['id'] == car_id]
            
            if not car_chunk.empty:
                car_data_all.append(car_chunk)
          
    # 如果没有找到该车辆的数据，继续下一个车辆
    if not car_data_all:
        print(f"未找到车辆ID {car_id} 的数据，跳过处理")
        continue
        
    # 合并该车辆的所有数据
    data = pd.concat(car_data_all, ignore_index=True)
    
    ###### 1. 数据处理
    ###### 1.1 数据格式整理
    # 删除不需要的列
    data = data.drop(['H','heading','shift_type'], axis=1)
    
     # 只保留2019年9月1日的数据
    data = data[data['date'] == '2019-09-01']
    
    # 如果筛选后数据为空，跳过该车辆
    if len(data) == 0 or data.empty:
        print(f"车辆ID {car_id} 在2019-09-01没有数据，跳过处理")
        continue
        
    # 转换时间格式
    data['date'] = data['date'].astype(str).str.zfill(8)  # 确保8位日期
    data['date'] = data['date'].str.slice(0,4) + '-' + data['date'].str.slice(4,6) + '-' + data['date'].str.slice(6,8)
    data['date_time'] = data['date_time'].astype(str).str.zfill(6)  # 确保6位时间
    data['date_time'] = data['date_time'].str.slice(0,2) + ':' + data['date_time'].str.slice(2,4) + ':' + data['date_time'].str.slice(4,6)
    data['time'] = pd.to_datetime(data['date'] + ' ' + data['date_time'],format='%Y-%m-%d %H:%M:%S')
    # 排序
    data = data.drop(['date_time','date'], axis=1)
    data.sort_values(by = ['id','time'],inplace = True)
    
    ###### 1.2 异常数据清洗
    data['pre_id'] = data['id'].shift(1)
    data['next_id'] = data['id'].shift(-1)
    data['pre_passenger'] = data['OpenStatus'].shift(1)
    data['next_passenger'] = data['OpenStatus'].shift(-1)
    # 剔除异常状态点  前后的载客状态相同但和当前不同，且ID一致
    data = data[-((data['id'] == data['pre_id']) & (data['id'] == data['next_id']) &
                 (data['pre_passenger'] == data['next_passenger']) & (data['OpenStatus'] != data['pre_passenger']))]
    data = data.drop(['pre_id', 'next_id', 'pre_passenger', 'next_passenger'], axis=1)
    
    ###### 2. GPS在线的识别充电事件（GPS online）
    ###### 2.1 漂移数据剔除
    data = tbd.traj_clean_redundant(data, col = ['id','time','lon','lat','speed'])
    # 区域外的数据剔除
    data = tbd.clean_outofshape(data, sz, col=['lon', 'lat'], accuracy=1000)
    # 区域内的漂移清洗，以速度、距离、角度三个方法进行清洗
    data = tbd.traj_clean_drift(data, col=['id', 'time', 'lon', 'lat'], speedlimit=80, dislimit=4000, anglelimit=40)
    
    ###### 2.2 停车与出行识别
    # 空间位移不超过1km持续10min
    # 定义栅格化参数 
    bounds = [113.75, 22.4, 114.62, 22.86]
    params = tbd.area_to_params(bounds, accuracy=1000)  
    ## 识别停车 停车阈值10min(600s) 位移1000m
    stay, move = tbd.traj_stay_move(data, params, col=['id', 'time', 'lon', 'lat'], activitytime=600) # 判断停车阈值10min(600s)
    
    # 检查stay是否为空
    if len(stay) == 0 or stay.empty:
        print(f"警告：车辆ID {car_id} 的stay数据框为空，跳过在线充电事件识别")
        continue
        
    stay['dis'] = tbd.getdistance(stay['lon'], stay['lat'], stay['lon'].shift(-1), stay['lat'].shift(-1))
    stay = stay[stay['dis'] <= 1000] # 位移不超过1000m
    
    # 再次检查筛选后的stay是否为空
    if len(stay) == 0 or stay.empty:
        print(f"车辆ID {car_id} 筛选位移后的stay数据框为空，跳过在线充电事件识别")
        continue
    
    ###### 2.3 充电站匹配
    ## 距离最近的充电站200m
    # 靠近充电站阈值（≤200米）
    try:
        stay = tbd.ckdnearest(stay, charge_station, Aname=['lon','lat'], Bname=['station_lon','station_lat']) # dist为距离充电站距离
        # 标记哪些停留在充电,dis距离阈值200m，duration停留时间阈值10min,600s
        stay['ischarge'] = ((stay['dist']<=200)& (stay['duration']>=600)).astype(int)
        # 如果不是充电，将充电站信息置空
        stay.loc[stay['ischarge']==0,['station_lon','station_lat','dist']]=np.nan
        online_charge = stay[(stay['ischarge'] == 1)].copy()
        
        # 清理不需要的列
        columns_to_drop = ['charge_station', 'LONCOL', 'LATCOL']
        for col in columns_to_drop:
            if col in online_charge.columns:
                online_charge = online_charge.drop([col], axis=1)
        
        # 添加充电类型标记
        online_charge['charge_type'] = 'online'
        
        # 在线充电事件添加到总列表
        if len(online_charge) > 0:
            all_charge_events.append(online_charge)
            print(f"车辆ID {car_id} 识别到 {len(online_charge)} 个在线充电事件")
    except Exception as e:
        print(f"处理车辆ID {car_id} 的在线充电事件时出错: {e}")
    
    ###### 3. 潜在的GPS离线的识别充电事件（GPS offline）
    # GPS关闭超过10分钟
    # 按时间排序
    data = data.sort_values('time')
    # 计算时间差
    data['next_time'] = data['time'].shift(-1)
    data['time_diff'] = (data['next_time'] - data['time']).dt.total_seconds()
    
    ## 找出时间差大于10分钟(600秒)的记录，GPS离线期间
    offline_periods = data[data['time_diff'] >= 600].copy()
    
    if len(offline_periods) == 0 or offline_periods.empty:
        print(f"车辆ID {car_id} 没有离线期间，跳过离线充电事件识别")
        continue
    
    # 计算离线前后的位置
    offline_periods['next_lon'] = offline_periods['lon'].shift(-1)
    offline_periods['next_lat'] = offline_periods['lat'].shift(-1)
    # 删除最后一行，因为它没有下一个点
    last_row = offline_periods.iloc[-1:].copy()
    offline_periods = offline_periods.dropna(subset=['next_lon', 'next_lat'])

    if len(offline_periods) == 0 or offline_periods.empty:
        print(f"车辆ID {car_id} 的离线期间数据不完整，跳过离线充电事件识别")
        continue

    ## 位移小于1000m
    # 计算离线前后的位移距离
    offline_periods['displacement'] = tbd.getdistance(
        offline_periods['lon'], offline_periods['lat'],
        offline_periods['next_lon'], offline_periods['next_lat']
    )
    
    # 筛选位移小于1000米的记录 
    potential_charge = offline_periods[offline_periods['displacement'] <= 1000].copy()

    if len(potential_charge) == 0 or potential_charge.empty:
        print(f"车辆ID {car_id} 没有符合条件的潜在充电事件，跳过离线充电事件识别")
        continue

    ## 距离最近的充电站200米以内
    try:
        # 计算离线前后点到最近充电站的距离
        # 离线前点到充电站的距离
        potential_charge = tbd.ckdnearest(
            potential_charge, charge_station,
            Aname=['lon', 'lat'],
            Bname=['station_lon', 'station_lat']
        )
        potential_charge.rename(columns={'dist': 'dist_off'}, inplace=True)
        
        # 离线后点到充电站的距离
        temp_df = pd.DataFrame({
            'lon': potential_charge['next_lon'],
            'lat': potential_charge['next_lat']
        })
        
        if len(temp_df) == 0 or temp_df.empty:
            print(f"车辆ID {car_id} 的temp_df数据框为空，跳过离线充电事件识别")
            continue
            
        temp_df = tbd.ckdnearest(
            temp_df, charge_station,
            Aname=['lon', 'lat'],
            Bname=['station_lon', 'station_lat']
        )
        potential_charge['dist_on'] = temp_df['dist']
        potential_charge['station_lon_on'] = temp_df['station_lon']
        potential_charge['station_lat_on'] = temp_df['station_lat']
        
        # 判断是否为充电事件 离线前或离线后点到充电站距离≤200米
        potential_charge['ischarge'] = ((potential_charge['dist_off'] <= 200) | 
                                      (potential_charge['dist_on'] <= 200)).astype(int)
        
        # 提取充电事件
        offline_charge = potential_charge[potential_charge['ischarge'] == 1].copy()
        
        if len(offline_charge) > 0:
            # 整理数据格式，与在线充电事件格式一致
            offline_charge['stime'] = offline_charge['time']
            offline_charge['etime'] = offline_charge['next_time']
            offline_charge['duration'] = offline_charge['time_diff']
            
            # 选择距离更近的充电站作为充电地点
            mask = offline_charge['dist_off'] <= offline_charge['dist_on']
            offline_charge.loc[mask, 'dist'] = offline_charge.loc[mask, 'dist_off']
            offline_charge.loc[~mask, 'dist'] = offline_charge.loc[~mask, 'dist_on']
            offline_charge.loc[mask, 'station_lon'] = offline_charge.loc[mask, 'station_lon']
            offline_charge.loc[~mask, 'station_lon'] = offline_charge.loc[~mask, 'station_lon_on']
            offline_charge.loc[mask, 'station_lat'] = offline_charge.loc[mask, 'station_lat']
            offline_charge.loc[~mask, 'station_lat'] = offline_charge.loc[~mask, 'station_lat_on']
            
            # 添加stayid列（使用索引作为唯一标识）
            offline_charge['stayid'] = offline_charge.index
            
            # 添加充电类型标记
            offline_charge['charge_type'] = 'offline'
            
            # 选择需要的列
            offline_charge = offline_charge[['id', 'stime', 'etime', 'lon', 'lat', 'duration', 'stayid',
                                             'station_lon', 'station_lat', 'dist', 'ischarge', 'charge_type']]
            
            # 将离线充电事件添加到列表
            all_charge_events.append(offline_charge)
            print(f"车辆ID {car_id} 识别到 {len(offline_charge)} 个离线充电事件")
        else:
            print(f"车辆ID {car_id} 没有识别到离线充电事件")
    except Exception as e:
        print(f"处理车辆ID {car_id} 的离线充电事件时出错: {e}")

end = time.perf_counter()    # 结束时间
print(f"运行时间: {end - start:.6f} 秒")

处理车辆:   0%|          | 1/21718 [00:48<293:05:03, 48.58s/it]

车辆ID 粤B0334Y 在2019-09-01没有数据，跳过处理


处理车辆:   0%|          | 2/21718 [01:35<287:16:15, 47.62s/it]

车辆ID 粤B0994Y 在2019-09-01没有数据，跳过处理


处理车辆:   0%|          | 3/21718 [02:23<287:29:14, 47.66s/it]

...

车辆ID 粤B4C8R2 在2019-09-01没有数据，跳过处理


处理车辆:   3%|▎         | 638/21718 [7:58:42<273:46:21, 46.75s/it]

车辆ID 粤B4C8R7 在2019-09-01没有数据，跳过处理


处理车辆:   3%|▎         | 638/21718 [7:58:57<263:45:00, 45.04s/it]


KeyboardInterrupt: 

In [ ]:
# 合并所有充电事件 
if all_charge_events:
    all_charge = pd.concat(all_charge_events, ignore_index=True)
    # 再次去除可能的重复事件（不同文件中可能有相同的车辆ID）
    all_charge = all_charge.sort_values('duration', ascending=False)
    all_charge = all_charge.drop_duplicates(subset=['id', 'stime'], keep='first')
    # 统计充电事件数量
    online_count = len(all_charge[all_charge['charge_type'] == 'online'])
    offline_count = len(all_charge[all_charge['charge_type'] == 'offline'])
    
    print('总充电事件数量：', len(all_charge))
    print('在线充电事件数量：', online_count)
    print('离线充电事件数量：', offline_count)
    
    # 保存结果
#     all_charge.to_csv('charging_events_result.csv', index=False)
#     print('充电事件结果已保存到charging_events_result.csv')
# else:
#     print('未识别到任何充电事件')